Agent Environment Loop

In [3]:
from typing import Any
import abc, time, numpy as np
import gymnasium as gym
import minigrid

Base Agent Class Interface

In [7]:
class Agent(abc.ABC):
    """Abstract Base Class defining standard RL Agent Interface"""
    
    @abc.abstractmethod
    def observe(self, observation: Any) -> None:
        """Receive and process the current environment state or observation"""
        pass

    @abc.abstractmethod
    def act(self) -> int:
        """Select and return an action based on the agent's current policy"""
        pass

    @abc.abstractmethod
    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        """Update internal weights, tables or history based on environment feedback"""
        pass

Initialise Random Agent

In [12]:
class RandomAgent(Agent):
    def __init__(self, action_space: gym.spaces.Discrete):
        self.action_space = action_space
    
    # A random agent does not observe neither does it update its internal state. It acts randomly.
    def observe(self, observation: Any) -> None:
        pass

    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        pass

    def act(self) -> int:
        return int(self.action_space.sample())

Initializing Cartpole Heuristics Agent

In [13]:
class CartpoleHeuristicsAgent(Agent):
    """Move the cart in the same direction the pole is tilting to balance it... as simple as that"""
    def __init__(self) -> None:
        self.pole_angle = 0.0
    
    def observe(self, observation: Any) -> None:
        # observation layout: [cart_pos, cart_vel, pole_angle, pole_vel]
        self.pole_angle = float(observation[2])
    
    # This agent uses fixed logic and does not learn
    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        pass

    def act(self) -> int:
        return 1 if self.pole_angle > 0 else 0

Initializing MiniGrid Heuristics Agent

In [15]:
class MiniGridHeuristicsAgent(Agent):
    """Rule: Move forward, if block by a wall, rotate right"""
    def __init__(self) -> None:
        self.is_front_clear: bool = True

    def observe(self, observation: Any) -> None:
        # observation['image'] - 7 x 7 image - space directly infront is at index (5, 3)
        grid = observation['image']
        front_cell = grid[5, 3]
        self.is_front_clear = (front_cell[0] != 2)
    
    def update(self, reward: float, terminated: bool, truncated: bool) -> None:
        pass

    def act(self) -> int:
        # 2 - forward, 1 - right, 0 - left
        return 2 if self.is_front_clear else 1